## This demo showcases the implementation of user story 530

This notebook shows the implementation of the different wrappers (rs-client-libraries) functions for the staging endpoints.

In [ ]:
import requests
import os
import pprint
import time
import pystac
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, stac_client, staging_client = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"    
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10

### Create a test collection named my_test_collection and check it afterwards to see if it's empty

In [ ]:
# Create a test collection 
collection = create_test_collection()
# Check the catalog for my_test_collection
items = stac_client.get_items(TEST_COLLECTION)
assert not list(items)

### Getting all the sessions from the input collection found in the configuration of CADIP station

In [ ]:
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0
for item in items_collection_cadip:
    print(f"Session {item.id} has {len(item.assets)} assets with datetime {item.properties.get('datetime')}")

### Getting all the assets from the input collection found in the configuration of ADGS station

In [ ]:
# Another example on how to get the items, will just load all the results in memory
# items_collection_adgs = list(auxip_client.get_items(adgs_collection_id))
items_collection_adgs = auxip_client.search(max_items = 14, collections = [adgs_collection_id])
assert len(items_collection_adgs) > 0

#pprint.PrettyPrinter(indent=4).pprint(items_collection_adgs)
for item in items_collection_adgs:
    print(f"AUXIP asset {item.id} has datetime {item.properties.get('datetime')}")

### Check existing processes + display information about the staging process

In [ ]:
# Returns list of all available processes from config.
processes = staging_client.get_processes()
print(processes)

In [ ]:
# Should return info about the staging process.
process_info = staging_client.get_process("staging")
print(process_info)

### Check the jobs table

In [ ]:
jobs = staging_client.get_jobs()
if jobs.get("numberMatched") > 0:
    delete_jobs = True
    if cluster_mode == True:
        delete_jobs = input(f"There are {jobs.get('numberMatched')} jobs in the table. Do you want to delete them all (y/n)?").lower().strip() == 'y'
    if delete_jobs:
        print("Deleting all the jobs...")
        for job in jobs.get("jobs"):
            delete_response = staging_client.delete_job(job.get('identifier'))
        # Check that the jobs have been deleted
        jobs = staging_client.get_jobs()
        print(f"Existing jobs: {jobs}")

### Starting 2 staging processes, one from the CADIP station and one from the ADGS station
The staging process from the ADGS station is expected to fail because one of the assets contains an incorrect download link for the file.

In [ ]:
staging_resp_list = []
for items in [pystac.ItemCollection(list(items_collection_cadip)), pystac.ItemCollection(list(items_collection_adgs))]:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))
    
timeout = 120
started_job_id_list = []

for resp in staging_resp_list:
    started_job_id_list.append(resp["status"]["running"])
    while timeout > 0:
        if "running" not in resp["status"]:
            break
        # TODO: to replace with the following commented line after the rs-server-staging update
        ###job_info = staging_client.get_job_info(resp["jobID"])
        job_info = staging_client.get_job_info(resp["status"]["running"])
        pprint.PrettyPrinter(indent=4).pprint(job_info)
        print("\n")
        if "successful" in job_info["status"]:
            print(" ----- Job COMPLETED \n")
            break
        if "failed" in job_info["status"]:
            print("-----Job FAILED \n")
            break
        time.sleep(2)
        timeout -= 2

### Check the catalog for my_test_collection. Ten items should be present now (one from CADIP station and nine from AUXIP station)

In [ ]:
# Check the catalog for my_test_collection
result = list(stac_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


### Check the jobs table

In [ ]:
# Check that each of the job previously launched are successful
for job_id in started_job_id_list:
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

### Delete the whole collection and recreate it

In [ ]:
# Create a test collection 
collection = create_test_collection()
# Check the catalog for my_test_collection
items = stac_client.get_items(TEST_COLLECTION)
assert not list(items)

### Stage some files from both cadip and auxip stations

In [ ]:
assert stage_test_objects(cadip_client, 18, objects_are_files = True)
assert stage_test_objects(auxip_client, 14, objects_are_files = True)